Collaborative filtering and content-based filtering are two core approaches in recommendation systems, each using different data to suggest items like movies or products.

Core Concepts
Collaborative filtering leverages user interactions across a group to recommend items, assuming users with similar past behaviors will like similar things in the future. Content-based filtering, by contrast, focuses on item features matching a user's individual profile, such as genre or keywords from their liked items.

How They Work
In collaborative filtering, systems analyze a user-item matrix of ratings or clicks to find similar users (user-based) or co-occurring items (item-based), then predict preferences. Content-based filtering builds user profiles from past interactions and computes similarity (e.g., via cosine similarity) to item attributes like descriptions or tags.

Key Comparison
Aspect	Collaborative Filtering	Content-Based Filtering
Aspect	Collaborative Filtering	Content-Based Filtering
Data Source	User behavior and interactions 
Item features/metadata 
Recommendations	From similar users' likes 
Similar to user's past items 
Strengths	Novel discoveries; no metadata needed 
Personalized; transparent 
Weaknesses	Cold start for new users/items 
Limited diversity 
Many systems combine both in hybrid models for better accuracy.

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [14]:
df = pd.read_csv("ecommerce_customers.csv")
df

,customer_id,age,gender,annual_income,spending_score,purchase_frequency,avg_order_value,total_purchases,browsing_time_minutes,product_category_preference,...,cross_category_purchases,repeat_purchase_rate,avg_session_duration,pages_per_session,geographic_region,preferred_shipping,support_interactions,product_reviews_count,social_shares,coupon_redemptions
0,1,58,Male,16639,93,16,92.35,16,157,Clothing,...,3,0.76,1008,3,South,Same Day,1,8,4,1
1,2,66,Female,21698,72,18,87.90,18,184,Home & Garden,...,1,0.93,1055,6,North,Same Day,1,14,5,2
2,3,28,Female,38283,76,16,106.05,16,227,Clothing,...,3,0.84,397,13,West,Same Day,2,14,1,2
3,4,26,Male,32219,93,18,111.93,18,191,Beauty,...,1,0.76,876,14,North,Overnight,2,10,0,2
4,5,45,Male,44735,70,16,114.16,16,237,Home & Garden,...,1,0.85,267,14,West,Standard,2,4,4,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,54,Male,29968,19,1,211.01,1,5,Beauty,...,1,0.49,302,11,North,Express,1,4,0,6
96,97,22,Female,33501,12,3,221.60,3,34,Sports,...,0,0.27,1153,8,North,Same Day,5,5,2,1
97,98,31,Male,42513,18,1,443.14,1,14,Beauty,...,0,0.32,422,7,East,Overnight,3,5,10,5
98,99,46,Female,30309,5,2,234.03,2,42,Home & Garden,...,1,0.36,331,4,South,Overnight,3,4,0,4


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 40 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   customer_id                  100 non-null    int64  
 1   age                          100 non-null    int64  
 2   gender                       100 non-null    object 
 3   annual_income                100 non-null    int64  
 4   spending_score               100 non-null    int64  
 5   purchase_frequency           100 non-null    int64  
 6   avg_order_value              100 non-null    float64
 7   total_purchases              100 non-null    int64  
 8   browsing_time_minutes        100 non-null    int64  
 9   product_category_preference  100 non-null    object 
 10  device_type                  100 non-null    object 
 11  last_purchase_days           100 non-null    int64  
 12  segment                      100 non-null    object 
 13  customer_lifetime_val

In [16]:
df.describe()

,customer_id,age,annual_income,spending_score,purchase_frequency,avg_order_value,total_purchases,browsing_time_minutes,last_purchase_days,customer_lifetime_value,...,email_open_rate,click_through_rate,cross_category_purchases,repeat_purchase_rate,avg_session_duration,pages_per_session,support_interactions,product_reviews_count,social_shares,coupon_redemptions
count,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.00000,...,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.00000,100.00000,100.000000
mean,50.500000,41.730000,31490.330000,50.500000,10.040000,183.646800,10.040000,107.410000,25.560000,2040.25200,...,0.235500,0.079600,1.470000,0.639500,662.830000,8.610000,1.580000,4.75000,4.05000,2.630000
std,29.011492,15.618466,10169.220141,31.976159,5.697634,98.881658,5.697634,74.109133,25.238787,1535.19896,...,0.176569,0.064931,1.029416,0.217439,307.888149,3.760225,1.248676,4.31201,4.01355,2.195749
min,1.000000,18.000000,15912.000000,3.000000,1.000000,85.420000,1.000000,5.000000,1.000000,59.76000,...,0.000000,0.000000,0.000000,0.210000,187.000000,3.000000,0.000000,0.00000,0.00000,0.000000
25%,25.750000,28.000000,22517.250000,17.750000,5.000000,105.162500,5.000000,42.000000,3.750000,723.69500,...,0.097500,0.030000,1.000000,0.447500,385.000000,5.000000,1.000000,1.00000,1.75000,1.000000
50%,50.500000,39.500000,30784.500000,43.500000,10.000000,160.745000,10.000000,87.000000,18.000000,1640.23000,...,0.180000,0.060000,1.000000,0.685000,637.000000,8.500000,2.000000,4.00000,3.00000,2.000000
75%,75.250000,54.250000,40469.250000,83.250000,15.000000,222.787500,15.000000,177.250000,41.000000,3191.90500,...,0.350000,0.122500,2.000000,0.830000,945.500000,12.000000,2.000000,6.25000,5.00000,3.000000
max,100.000000,70.000000,49962.000000,100.000000,20.000000,447.960000,20.000000,240.000000,84.000000,5820.26000,...,0.650000,0.290000,4.000000,0.950000,1198.000000,15.000000,5.000000,15.00000,16.00000,8.000000


In [17]:
df.isnull().sum()

customer_id                    0
age                            0
gender                         0
annual_income                  0
spending_score                 0
purchase_frequency             0
avg_order_value                0
total_purchases                0
browsing_time_minutes          0
product_category_preference    0
device_type                    0
last_purchase_days             0
segment                        0
customer_lifetime_value        0
return_rate                    0
payment_method                 0
newsletter_subscribed          0
social_media_engagement        0
avg_review_rating              0
cart_abandonment_rate          0
discount_usage_pct             0
referral_source                0
customer_satisfaction_score    0
preferred_shopping_hour        0
mobile_app_user                0
wishlist_items                 0
customer_since_months          0
loyalty_tier                   0
email_open_rate                0
click_through_rate             0
cross_cate

In [18]:
df.duplicated().sum()

np.int64(0)

In [19]:
#build a simple similarity based recommendation system using cosine similarity
from sklearn.metrics.pairwise import cosine_similarity
#drop the customer id and name columns
df.drop(["customer_id"], axis=1, inplace=True)

In [22]:
#converting all object to numerical values using label encoding
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = le.fit_transform(df[col])   

In [23]:
#calculate the cosine similarity between the customers
similarity = cosine_similarity(df)
similarity
#recommend the top 5 similar customers for each customer
similarity_df = pd.DataFrame(similarity, index=df.index, columns=df.index)
similarity_df
def recommend(customer_id):
    similar_customers = similarity_df[customer_id].sort_values(ascending=False).index[1:6]
    return similar_customers
recommend(0)

Index([14, 39, 30, 1, 9], dtype='int64')

In [24]:
#after getting the similar customers we can recommend products that they have bought but the current customer has not bought
def recommend_products(customer_id):
    similar_customers = recommend(customer_id)
    products = []
    for customer in similar_customers:
        products.extend(df.loc[customer, "product_category"].split(","))
    products = list(set(products))
    return products
recommend_products(0)

KeyError: 'product_category'